# WikiLens · try the product

Run the cells in order with the project Python3.12 kernel. On SSH select **Junction T4 Python3.12**. Live search runs the current HyDE → BGE-M3 → MariaDB S → heading chunks/BGE reranker → editor article → paragraph citation pipeline, keeping its models loaded. Saved run replays actual captures and needs no GPU or database.

The answer supports multiple citations per paragraph. Click a citation to focus its exact quotation, or expand the original passage and open its Wikipedia revision. These are unreviewed drafts; quotations prove provenance, not every factual claim.

**Private SSH access:** forward port7860 in VS Code, or run `ssh -N -L 7860:127.0.0.1:7860 junction-hf-t4` on your laptop, then open http://127.0.0.1:7860. No public Gradio share link is created. The temporary SSH job has its own fixed timeout; this notebook never starts or extends paid compute.

You may also run the notebook locally and set `REMOTE_URL` to a forwarded remote Gradio address; otherwise local mode supports saved playback. If port7860 already serves WikiLens, the launch cell attaches to that warm app.

A laptop proxy can connect to the private SSH GPU app while that temporary job remains active. Request timings exclude initial model loading; setup never starts a paid job.

Reader context now guides an evidence relevance filter before generation. Expand pipeline details to inspect selected/excluded source IDs and the focused question used downstream. Saved runs retain their recorded context and cannot be relabelled with a new one.

Current cutoff experiment uses **S:29,460 articles in Finnish, English and Swedish**, selected in the settings cell below. Full-L source files and the earlier checkpoint report are preserved locally; its temporary database needs rebuilding after the earlier job ended. Do not select L until its import completes. Answer language is independent of source language. Advanced settings expose score cutoffs and up to25 answer passages by default. Focus selection is batched; source passages are packed whole against the actual generation-token budget.


In [ ]:
from pathlib import Path
import sys, os
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').exists() and (p / 'wikilens').is_dir())
PORT = 7860
CORPUS_DB = 'wikilens_s_cutoff'  # S for cutoff study. Full L later: 'wikilens' after its import completes.
if str(ROOT).startswith('/workspace/'):
    os.environ['DB_NAME'] = CORPUS_DB  # This kernel only; .env remains unchanged.
MODE = 'Live search' if str(ROOT).startswith('/workspace/') else 'Saved run (offline)'
os.environ['GRADIO_ANALYTICS_ENABLED'] = 'False'
print('Project:', ROOT, '\nMode:', MODE)
REMOTE_URL = None  # Optional on laptop: 'http://127.0.0.1:17860' via ssh -L 17860:127.0.0.1:7860 junction-hf-t4


In [ ]:
# Installs only the locked project + demo dependency group; no model training.
import shutil, subprocess
uv = shutil.which('uv') or next((str(p) for p in [Path('/workspace/.uv-bootstrap/bin/uv'), Path.home()/'.local/bin/uv'] if p.exists()), None)
assert uv, 'Install uv, then run uv sync --locked --group demo in the project.'
assert sys.version_info[:2] == (3, 12), 'Select the project Python 3.12 kernel.'
subprocess.run([uv, 'sync', '--locked', '--group', 'demo'], cwd=ROOT, check=True)


In [ ]:
# Reuse this product UI when it is already running, preserving warm models.
sys.path.insert(0, str(ROOT))
import requests
from IPython.display import display, IFrame
from wikilens.gradio_demo import build_demo, launch_demo
try:
    existing = requests.get(f'http://127.0.0.1:{PORT}/config', timeout=1).json()
except (requests.RequestException, ValueError):
    existing = None
if existing and existing.get('title') == 'WikiLens · Wikipedia evidence':
    print(f'Using the existing UI: http://127.0.0.1:{PORT}. Forward this port when on SSH.')
    display(IFrame(f'http://127.0.0.1:{PORT}', width='100%', height=900))
else:
    if 'demo' in globals(): demo.close()
    if 'controller' in globals(): controller.close()
    demo, controller = build_demo(default_mode=MODE, remote_url=REMOTE_URL)
    launch_demo(demo, port=PORT, inline=True)


## Try it

1. Select **Talvisota · suomeksi**, then **Explore Wikipedia**. In live mode this performs a new search; saved mode is labelled with its capture timestamp.
2. Click a citation and inspect the quotation, full source passage and revision.
3. In live mode change the question, perspective or source editions and submit again. Adjust answer length under settings.
4. Expand **HyDE, retrieval and timings**; HyDE is only a search aid.
5. Use Reset to clear the interface. Models stay warm between questions.

Live calls save full prompts, outputs, source metadata, timings and intermediate captures to ignored `data/gradio-runs/`. Initial model loading is slower than subsequent questions. No new cloud jobs or charges are created by setup.


In [ ]:
# Optional: uncomment when finished to stop the UI and release this notebook's models.
# demo.close()
# controller.close()
